In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, GroupShuffleSplit, GroupKFold
from sklearn.decomposition import PCA
from scipy.stats import randint
from sklearn.metrics import precision_score, recall_score, accuracy_score
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
def get_condition(data_with_conditions, row_name='Diet'):
    '''
    Extract diet information from the metadata file.
    
    Inputs:
        data_with_conditions : str
            Path to the metadata CSV file containing 'Diet' and 'OmicsEarTag' rows.
        row : int (unused, kept for backward compatibility)
            Placeholder parameter (can be ignored).
    
    Outputs:
        pd.DataFrame - Clean mapping of FAC_ID to Diet
    '''
    # Load metadata
    meta = pd.read_csv(data_with_conditions, index_col=0)
    
    # Extract relevant rows 
    diet_row = meta.loc[row_name]
    fac_row = meta.loc['OmicsEarTag']
    
    # Build mapping
    diet_map = pd.DataFrame({
        'FAC_ID': fac_row.values,
        row_name: diet_row.values
    })
    
    # Drop unwanted rows (header-like entries)
    diet_map = diet_map[~diet_map['FAC_ID'].isin(['Orig_ID', 'IDFemaleAgingColony'])].reset_index(drop=True)
    
    return diet_map

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv") # load diet data

In [ ]:
diet_data.to_csv('../diet_data.csv') # save for later use

In [ ]:
sampling_data_100 = pd.read_csv('../100_combined_samples.csv') # load flux matrix

In [ ]:
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)] # remove where all columns (reactions) are zero

# spliting per group

In [ ]:
def prepare_Xy_groups_by_mouse(
    X_df: pd.DataFrame,
    y_df: pd.DataFrame,
    id_col: str = "Unnamed: 0",
    y_col: str = "Diet",
    mouse_regex: str = r"(FAC\d+)",   # regex to extract mouse ID from sample string
    drop_all_zero_cols: bool = False, # drop features that are all zeros
    verbose: bool = True,
):
    """
    Prepares features, labels, and groups for mouse-level grouped analysis.
    Extracts mouse ID from sample identifiers, merges with labels, and returns split data.
    
    Parameters:
    -----------
    X_df : DataFrame
        Feature matrix with sample IDs in id_col
    y_df : DataFrame
        Label lookup table with FAC_ID and y_col
    id_col : str
        Column name in X_df containing sample identifiers
    y_col : str
        Target column name in y_df (e.g., "Diet")
    mouse_regex : str
        Regex pattern to extract mouse ID from sample ID strings
    drop_all_zero_cols : bool
        If True, remove features that are all zeros
    verbose : bool
        If True, print summary statistics
    
    Returns:
    --------
    X : DataFrame
        Feature matrix
    y : Series
        Labels
    groups : Series
        Mouse IDs for grouping
    """
    
    # Validate required columns exist
    if id_col not in X_df.columns:
        raise ValueError(f"{id_col} not found in X_df")
    if "FAC_ID" not in y_df.columns:
        raise ValueError("FAC_ID not found in y_df")
    if y_col not in y_df.columns:
        raise ValueError(f"{y_col} not found in y_df")

    Xw = X_df.copy()

    # Extract mouse ID from sample identifier using regex
    Xw["mouse_id"] = Xw[id_col].astype(str).str.extract(mouse_regex, expand=False)
    if Xw["mouse_id"].isna().any():
        bad = Xw.loc[Xw["mouse_id"].isna(), id_col].head(5).tolist()
        raise ValueError(f"Could not extract mouse_id from {id_col}. Examples: {bad}")

    # Merge mouse-level labels onto feature rows
    y_small = y_df[["FAC_ID", y_col]].drop_duplicates("FAC_ID", keep="first").copy()
    y_small = y_small.rename(columns={"FAC_ID": "mouse_id"})

    merged = Xw.merge(y_small, on="mouse_id", how="inner")
    if merged.empty:
        raise ValueError("No matching mouse IDs between X_df and y_df")

    # Extract labels and group identifiers
    y = merged[y_col].reset_index(drop=True)
    groups = merged["mouse_id"].reset_index(drop=True)

    # Extract features: drop metadata columns (id, mouse_id, label)
    drop_cols = [id_col, "mouse_id", y_col]
    X = merged.drop(columns=[c for c in drop_cols if c in merged.columns])
    X = X.apply(pd.to_numeric, errors="coerce")

    # Optionally remove features with no variation
    if drop_all_zero_cols:
        all_zero = (X.fillna(0) == 0).all(axis=0)
        X = X.loc[:, ~all_zero]

    X = X.reset_index(drop=True)

    # Print summary
    if verbose:
        print(f"Matched rows: {len(merged)}")
        print(f"Unique mice: {groups.nunique()}")
        print(f"Final shapes: X={X.shape}, y={y.shape}, groups={groups.shape}")

    return X, y, groups


In [ ]:
sampling_data_100 = pd.read_csv('../100_combined_samples.csv')
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)]

In [ ]:
diet_data = pd.read_csv('../diet_data.csv') # load diet data

In [ ]:
X, y, groups = prepare_Xy_groups_by_mouse(X_df=sampling_data_100, y_df=diet_data) # make groups

In [ ]:
def optimize_random_forest_with_PCA_toggle_with_scaler_toggle_groups(
    X,
    y,
    groups,
    pca_toggle=False,
    scaler_toggle=False,
    n_iter=10,
    n_PCs=45,
    random_state=4,
    printing=False
):
    '''
    Optimizes a random forest classifier, optionally applying PCA and/or scaling,
    using GROUP-AWARE splitting and CV (mouse-level groups) to avoid leakage.

    Inputs:
        X: pandas.DataFrame
            Feature matrix (run-level rows).
        y: pandas.Series or array-like
            Target labels (same length as X).
        groups: pandas.Series or array-like
            Group labels (e.g., mouse_id) same length as X. Must have 275 unique mice.
        pca_toggle: bool
            Whether to include PCA dimensionality reduction in the pipeline.
        scaler_toggle: bool
            Whether to include StandardScaler in the pipeline.
    Returns:
        random_search: RandomizedSearchCV
        X_test, y_test
        feature_importances (DataFrame or None)
    '''

    # Group-aware train/test split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]

    # Build pipeline 
    steps = []
    if scaler_toggle:
        steps.append(('scaler', StandardScaler()))
    if pca_toggle:
        steps.append(('pca', PCA()))
    steps.append(('rf', RandomForestClassifier(random_state=42)))
    pipeline = Pipeline(steps)

    # Hyperparameter space
    hyperparameter_space = {
        'rf__n_estimators': randint(50, 100),
        'rf__max_depth': [10, 20, 30],
        'rf__min_samples_split': randint(2, 10),
        'rf__min_samples_leaf': randint(1, 5)
    }

    if pca_toggle:
        hyperparameter_space['pca__n_components'] = [n_PCs]

    # Group-aware CV
    # Prefer StratifiedGroupKFold if available (keeps class balance per fold)
    # Fallback to GroupKFold otherwise.
    try:
        from sklearn.model_selection import StratifiedGroupKFold
        cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=random_state)
        cv_splitter = cv.split(X_train, y_train, groups=groups_train)
        cv_for_search = list(cv_splitter)  # RandomizedSearchCV accepts list of splits
    except Exception:
        cv = GroupKFold(n_splits=4)
        cv_splitter = cv.split(X_train, y_train, groups=groups_train)
        cv_for_search = list(cv_splitter)


    # Randomized search (group-aware CV splits)
    random_search = RandomizedSearchCV(
        pipeline,
        param_distributions=hyperparameter_space,
        n_iter=n_iter,
        cv=cv_for_search,
        n_jobs=4,  # to keep RAM usage safe
        verbose=1,
        random_state=2,
        scoring='f1_macro'
    )

    random_search.fit(X_train, y_train)

    if printing:
        print(f"Best Parameters: {random_search.best_params_}")
        print(f"Best CV Score: {random_search.best_score_:.4f}")

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(X_test)

    if printing:
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        # sanity check: no mouse overlap between train and test
        train_mice = set(groups_train)
        test_mice = set(groups.iloc[test_idx])
        print("\nGroup sanity check:")
        print("Train unique mice:", len(train_mice))
        print("Test unique mice:", len(test_mice))
        print("Overlap (should be 0):", len(train_mice.intersection(test_mice)))

    # Feature importance (only if PCA not used)
    feature_importances = None
    if not pca_toggle:
        rf_model = best_model.named_steps['rf']
        feature_importances = pd.DataFrame({
            'Feature': X.columns,
            'Importance': rf_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)

        if printing:
            print("\nTop 10 Most Important Features:")
            print(feature_importances.head(10))
    else:
        if printing:
            print("\nPCA enabled - feature importances are not available.")

    return random_search, X_test, y_test, feature_importances


In [ ]:
def repeat_rf_optimization_with_scaler_toggle_groups(
    X,
    y,
    groups,
    n_runs=50,
    pca_toggle=False,
    scaler_toggle=False,
    n_iter=10,
    n_PCs=25
):
    '''
    Repeatedly runs RF optimization with group-aware splitting and CV,
    collecting confusion matrices, scores, and feature importances.

    Returns:
        confusions: list of confusion matrices
        scores: list of F1-macro scores
        avg_confusion: mean confusion matrix
        avg_importances: mean feature importance (or None if PCA enabled)
    '''

    # Initialize lists to collect results from each run
    confusions = []
    scores = []
    importances_list = []
    accuracies = []
    recalls = []
    precisions = []

    y_tests = []
    y_preds = []

    # Run optimization multiple times with different random seeds
    for i in range(n_runs):

        print(f"Run {i+1}/{n_runs}\n")

        # Optimize RF with group-aware CV and collect test predictions
        rs, X_test, y_test, feature_importances = optimize_random_forest_with_PCA_toggle_with_scaler_toggle_groups(
            X, y, groups, pca_toggle=pca_toggle, scaler_toggle=scaler_toggle, n_iter=n_iter, n_PCs=n_PCs, random_state=i + 42, printing=False
        )

        best_model = rs.best_estimator_
        y_pred = best_model.predict(X_test)

        # Ensure y_test and y_pred are pandas Series with aligned indices
        if not isinstance(y_test, pd.Series):
            y_test = pd.Series(y_test, index=X_test.index, name="y_test")
        else:
            y_test = y_test.loc[X_test.index]
            y_test.name = "y_test"

        y_pred = pd.Series(y_pred, index=X_test.index, name="y_pred")

        y_tests.append(y_test.copy())
        y_preds.append(y_pred.copy())

        # Compute and collect metrics for this run
        cm = confusion_matrix(y_test, y_pred)
        confusions.append(cm)

        f1 = f1_score(y_test, y_pred, average="macro")
        scores.append(f1)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

        recall = recall_score(y_test, y_pred, average="macro")
        recalls.append(recall)

        precision = precision_score(y_test, y_pred, average="macro")
        precisions.append(precision)

        # Collect feature importances (only if PCA not used)
        if not pca_toggle:
            if isinstance(feature_importances, pd.DataFrame):
                s = feature_importances.set_index("Feature")["Importance"].astype(float)
            else:
                s = pd.Series(np.asarray(feature_importances, dtype=float), index=X.columns)

            importances_list.append(s)

    # Calculate average metrics across all runs
    avg_confusion = np.mean(confusions, axis=0)

    print("\n=======================================")
    print("Average Confusion Matrix Across Runs")
    print("=======================================")
    print(avg_confusion)

    # Print performance summary
    print("\n=======================================")
    print("Performance Summary")
    print("=======================================")
    print(f"Mean F1-macro: {np.mean(scores):.4f}")
    print(f"Std  F1-macro: {np.std(scores):.4f}")
    print(f"Mean accuracy-macro: {np.mean(accuracies):.4f}")
    print(f"Std  accuracy-macro: {np.std(accuracies):.4f}")
    print(f"Mean recall-macro: {np.mean(recalls):.4f}")
    print(f"Std  recall-macro: {np.std(recalls):.4f}")
    print(f"Mean precision-macro: {np.mean(precisions):.4f}")
    print(f"Std  precision-macro: {np.std(precisions):.4f}")

    # Average feature importances if PCA not enabled
    if not pca_toggle:
        importances_df = pd.concat(importances_list, axis=1).fillna(0.0)
        avg_s = importances_df.mean(axis=1)

        avg_importances = (avg_s.sort_values(ascending=False)
                        .reset_index()
                        .rename(columns={"index": "Feature", 0: "AvgImportance"}))

        print("\n=======================================")
        print("Average Feature Importance Across Runs")
        print("=======================================")
        print(avg_importances.head(20))
    else:
        avg_importances = None
        print("\nPCA enabled — skipping feature importance aggregation.")

    return (
        confusions,
        scores,
        accuracies,
        recalls,
        precisions,
        y_tests,
        y_preds,
        avg_confusion,
        avg_importances
    )


## diet

In [ ]:
complete_data_diet = repeat_rf_optimization_with_scaler_toggle_groups(X, y, groups, n_runs=30, n_iter=10)

In [ ]:
(confusions_diet, scores_diet, accuracies_diet, recalls_diet, precisions_diet,
 y_tests_diet, y_preds_diet,
 avg_confusion_diet, avg_importances_diet) = complete_data_diet

# save per-run confusion matrices
np.save("../final_result/diet_pred_flux/confusions_diet.npy", np.array(confusions_diet))

# save per-run metrics
metrics_per_run = pd.DataFrame({
    "run": np.arange(1, len(scores_diet) + 1),
    "f1_macro": scores_diet,
    "accuracy_macro": accuracies_diet,
    "recall_macro": recalls_diet,
    "precision_macro": precisions_diet
})
metrics_per_run.to_csv("../final_result/diet_pred_flux/metrics_per_run.csv", index=False)

# Instead of np.save, stack them into a readable CSV that keeps indices.
print("Stacking predictions for safe storage...")
all_preds_rows = []
for run_id, (yt, yp) in enumerate(zip(y_tests_diet, y_preds_diet), start=1):
    # Create a mini-dataframe for this run; yt.index preserves the Sample ID
    df_run = pd.DataFrame({
        "y_true": yt.values,
        "y_pred": yp.values if hasattr(yp, "values") else yp,
        "run": run_id
    }, index=yt.index) # <--- This safeguards Sample/Mouse IDs
    all_preds_rows.append(df_run)

# Concatenate and save
all_predictions_df = pd.concat(all_preds_rows)
all_predictions_df.to_csv("../final_result/diet_pred_flux/all_predictions_detailed.csv", index=True)

# save average confusion matrix
pd.DataFrame(avg_confusion_diet).to_csv("../final_result/diet_pred_flux/avg_confusion_diet.csv", index=False)

# save average feature importances (if PCA off)
if avg_importances_diet is not None:
    avg_importances_diet.to_csv("../final_result/diet_pred_flux/avg_importances_diet.csv", index=False)

In [ ]:
all_predictions_df

## sex

In [ ]:
sex_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Sex')
sampling_data_100 = pd.read_csv('../100_combined_samples.csv')
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)]
X, y, groups = prepare_Xy_groups_by_mouse(X_df=sampling_data_100, y_df=sex_data, y_col='Sex')
del sampling_data_100
del sex_data
results_sex = repeat_rf_optimization_with_scaler_toggle_groups(X, y, groups)

In [ ]:
(confusions_sex, scores_sex, accuracies_sex, recalls_sex, precisions_sex,
 y_tests_sex, y_preds_sex,
 avg_confusion_sex, avg_importances_sex) = results_sex

# save per-run confusion matrices
np.save("../final_result/confusions_sex.npy", np.array(confusions_sex))

# save per-run metrics
metrics_per_run = pd.DataFrame({
    "run": np.arange(1, len(scores_sex) + 1),
    "f1_macro": scores_sex,
    "accuracy_macro": accuracies_sex,
    "recall_macro": recalls_sex,
    "precision_macro": precisions_sex
})
metrics_per_run.to_csv("../final_result/metrics_per_run_sex.csv", index=False)

# save all y_tests and y_preds
np.save("../final_result/y_tests_sex.npy", y_tests_sex)
np.save("../final_result/y_preds_sex.npy", y_preds_sex)
# save average confusion matrix
pd.DataFrame(avg_confusion_sex).to_csv("../final_result/avg_confusion_sex.csv", index=False)

# save average feature importances (if PCA off)
if avg_importances_sex is not None:
    avg_importances_sex.to_csv("../final_result/avg_importances_sex.csv", index=False)

## age

In [ ]:
age_data = pd.read_csv('../age_info.csv', index_col=0)
age_data

In [ ]:
sampling_data_100 = pd.read_csv('../100_combined_samples.csv')
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)]
X, y, groups = prepare_Xy_groups_by_mouse(X_df=sampling_data_100, y_df=age_data, y_col='Age_Group')
del sampling_data_100
del age_data
results_age = repeat_rf_optimization_with_scaler_toggle_groups(X, y, groups)

In [ ]:
(confusions_age, scores_age, accuracies_age, recalls_age, precisions_age,
 y_tests_age, y_preds_age,
 avg_confusion_age, avg_importances_age) = results_age

# save per-run confusion matrices
np.save("../final_result/confusions_age.npy", np.array(confusions_age))

# save per-run metrics
metrics_per_run = pd.DataFrame({
    "run": np.arange(1, len(scores_age) + 1),
    "f1_macro": scores_age,
    "accuracy_macro": accuracies_age,
    "recall_macro": recalls_age,
    "precision_macro": precisions_age
})
metrics_per_run.to_csv("../final_result/metrics_per_run_age.csv", index=False)

# save all y_tests and y_preds
np.save("../final_result/y_tests_age.npy", y_tests_age)
np.save("../final_result/y_preds_age.npy", y_preds_age)
# save average confusion matrix
pd.DataFrame(avg_confusion_age).to_csv("../final_result/avg_confusion_age.csv", index=False)

# save average feature importances (if PCA off)
if avg_importances_age is not None:
    avg_importances_age.to_csv("../final_result/avg_importances_age.csv", index=False)

## Combinging mice

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
# Load Data
y_tests_diet = np.load("../final_result/diet_pred_flux/y_tests_diet.npy", allow_pickle=True)
y_preds_diet = np.load("../final_result/diet_pred_flux/y_preds_diet.npy", allow_pickle=True)

# 'groups' must be defined previously

# Define Helper Functions
def stack_runs(y_tests, y_preds, groups):
    """
    Combines predictions from multiple runs into a single long-format DataFrame.
    """
    rows = []
    # Iterate through each run (fold)
    for run, (yt, yp) in enumerate(zip(y_tests, y_preds), start=1):
        df = pd.DataFrame({"y_true": yt, "y_pred": yp})
        df["run"] = run
        
        # Ensure 'groups' is sliced correctly corresponding to this specific run.
        df["mouse_id"] = groups.loc[df.index].values 
        
        rows.append(df)
    return pd.concat(rows, axis=0)

def majority_vote(s):
    """Returns the most frequent value in a series."""
    return s.value_counts().idxmax()

# Create Long-Format Data
pred_long = stack_runs(y_tests_diet, y_preds_diet, groups)

In [ ]:
# Specify index_col=0 to ensure the Sample IDs (indices) are loaded correctly
pred_long = pd.read_csv("../final_result/diet_pred_flux/all_predictions_detailed.csv", index_col=0)

# Map Mouse IDs (if they weren't saved in the CSV previously)
if "mouse_id" not in pred_long.columns:
    # Ensure 'groups' is defined and aligns with these indices
    pred_long["mouse_id"] = groups.loc[pred_long.index].values

print(pred_long.head())

In [ ]:
# Define Majority Vote Function
def majority_vote(s):
    return s.mode()[0]

# Perform Level 1 Aggregation (Per Run, Per Mouse)
# We aggregate BOTH predictions and true labels to ensure alignment
mouse_run_df = (pred_long
                .groupby(["run", "mouse_id"])
                .agg({
                    "y_pred": majority_vote,  # Vote on prediction
                    "y_true": "first"         # True label is constant for a mouse in a run
                })
                .reset_index())

# Generate One Confusion Matrix Per Run
confusions_per_run_mouse_level = []
runs = sorted(mouse_run_df["run"].unique())

for r in runs:
    # Filter for the current run
    run_data = mouse_run_df[mouse_run_df["run"] == r]
    
    # Calculate confusion matrix for this run (using aggregated mouse labels)
    cm = confusion_matrix(run_data["y_true"], run_data["y_pred"])
    
    confusions_per_run_mouse_level.append(cm)

# Verification 
print(f"Generated {len(confusions_per_run_mouse_level)} confusion matrices.")
print("\nExample: Confusion Matrix for Run 1 (Mouse Level):")
print(confusions_per_run_mouse_level[0])

In [ ]:
np.save("../final_result/confusions_diet_mouse_level.npy", np.array(confusions_per_run_mouse_level))

In [ ]:
def get_majority_vote(x):
    return x.mode()[0]

# Group by Run + Mouse to get 1 row per mouse per run
mouse_level_data = (pred_long
                    .groupby(["run", "mouse_id"])
                    .agg({
                        "y_true": "first",           # True label is constant
                        "y_pred": get_majority_vote  # Majority vote prediction
                    })
                    .reset_index())

# Compute Metrics for Each Run
mouse_scores = []
mouse_accuracies = []
mouse_recalls = []
mouse_precisions = []

# Get list of run numbers (e.g., 1 to 50)
run_numbers = sorted(mouse_level_data["run"].unique())

for r in run_numbers:
    # Filter data for just this specific run
    run_df = mouse_level_data[mouse_level_data["run"] == r]
    
    y_true_run = run_df["y_true"]
    y_pred_run = run_df["y_pred"]
    
    # Calculate metrics at the MOUSE level
    # We use 'macro' average to treat all classes equally
    f1 = f1_score(y_true_run, y_pred_run, average="macro")
    acc = accuracy_score(y_true_run, y_pred_run)
    rec = recall_score(y_true_run, y_pred_run, average="macro")
    prec = precision_score(y_true_run, y_pred_run, average="macro", zero_division=0)
    
    mouse_scores.append(f1)
    mouse_accuracies.append(acc)
    mouse_recalls.append(rec)
    mouse_precisions.append(prec)

# Create DataFrame and Save
metrics_per_run_mouse = pd.DataFrame({
    "run": run_numbers,
    "f1_macro": mouse_scores,
    "accuracy_macro": mouse_accuracies,
    "recall_macro": mouse_recalls,
    "precision_macro": mouse_precisions
})

print("Preview of Mouse-Level Metrics per Run:")
print(metrics_per_run_mouse.head())

# Save to CSV
metrics_per_run_mouse.to_csv("../final_result/metrics_per_run_mouse_level.csv", index=False)
print("\nSaved successfully to ../final_result/metrics_per_run_mouse_level.csv")

# Predicting mouse identity

In [ ]:
def repeat_rf_optimization_with_scaler_toggle_final(
    X,
    y,
    n_runs=50,
    scaler_toggle=False,
    n_iter=10,
    n_PCs=25,
    random_state=42
):
    '''
    Repeatedly trains and optimizes Random Forest models with different random seeds,
    collecting performance metrics and feature importances across multiple runs.
    
    This function performs multiple independent train-test splits, hyperparameter
    optimization, and prediction. Results are aggregated to compute mean and std
    of performance metrics and average feature importance.

    Parameters:
    -----------
    X : DataFrame
        Feature matrix (rows=samples, columns=features)
    y : array-like
        Target labels (1D)
    n_runs : int
        Number of times to repeat the optimization (default=50)
    scaler_toggle : bool
        If True, apply scaling (StandardScaler) to features (default=False)
    n_iter : int
        Number of hyperparameter configurations to try in RandomizedSearchCV (default=10)
    n_PCs : int
        Number of principal components if PCA is used (default=25)
    random_state : int
        Base random seed; each run uses random_state + i (default=42)

    Returns:
    --------
    confusions : list of arrays
        Confusion matrices from each run
    scores : list of float
        F1-macro scores from each run
    accuracies : list of float
        Accuracy scores from each run
    recalls : list of float
        Recall-macro scores from each run
    precisions : list of float
        Precision-macro scores from each run
    y_tests : list of Series
        True labels for test set in each run (aligned to X_test.index)
    y_preds : list of Series
        Predicted labels for test set in each run (aligned to X_test.index)
    avg_confusion : array
        Mean confusion matrix across all runs
    avg_importances : DataFrame
        Mean feature importance across all runs, sorted by importance (descending)
    '''

    # Initialize lists to collect results from each run
    confusions = []
    scores = []
    importances_list = []
    accuracies = []
    recalls = []
    precisions = []

    y_tests = []
    y_preds = []

    # Convert y to flat numpy array for consistent stratification
    if hasattr(y, "to_numpy"):
        y_arr = y.to_numpy()
    else:
        y_arr = y
    y_arr = np.asarray(y_arr).ravel()

    # Run optimization multiple times with different random seeds
    for i in range(n_runs):
        print(f"Run {i+1}/{n_runs}\n")

        # Optimize RF with scaler toggle and collect test predictions
        model, X_test, y_test, feature_importances = optimize_random_forest_with_PCA_toggle_with_scaler_toggle_final(
            X,
            y_arr,
            scaler_toggle=scaler_toggle,
            random_state=i + random_state,
            printing=False
        )

        best_model = model
        y_pred = best_model.predict(X_test)

        # Ensure y_test and y_pred are pandas Series with aligned indices
        if not isinstance(y_test, pd.Series):
            y_test = pd.Series(y_test, index=X_test.index, name="y_test")
        else:
            y_test = y_test.loc[X_test.index]
            y_test.name = "y_test"

        y_pred = pd.Series(y_pred, index=X_test.index, name="y_pred")

        y_tests.append(y_test.copy())
        y_preds.append(y_pred.copy())

        # Compute and collect metrics for this run
        cm = confusion_matrix(y_test, y_pred)
        confusions.append(cm)

        f1 = f1_score(y_test, y_pred, average="macro")
        scores.append(f1)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

        recall = recall_score(y_test, y_pred, average="macro")
        recalls.append(recall)

        precision = precision_score(y_test, y_pred, average="macro")
        precisions.append(precision)

        # Collect feature importances
        if isinstance(feature_importances, pd.DataFrame):
            s = feature_importances.set_index("Feature")["Importance"].astype(float)
        else:
            s = pd.Series(np.asarray(feature_importances, dtype=float), index=X.columns)

        importances_list.append(s)

    # Calculate average metrics across all runs
    avg_confusion = np.mean(confusions, axis=0)

    print("\n=======================================")
    print("Average Confusion Matrix Across Runs")
    print("=======================================")
    print(avg_confusion)

    # Print performance summary
    print("\n=======================================")
    print("Performance Summary")
    print("=======================================")
    print(f"Mean F1-macro: {np.mean(scores):.4f}")
    print(f"Std  F1-macro: {np.std(scores):.4f}")
    print(f"Mean accuracy-macro: {np.mean(accuracies):.4f}")
    print(f"Std  accuracy-macro: {np.std(accuracies):.4f}")
    print(f"Mean recall-macro: {np.mean(recalls):.4f}")
    print(f"Std  recall-macro: {np.std(recalls):.4f}")
    print(f"Mean precision-macro: {np.mean(precisions):.4f}")
    print(f"Std  precision-macro: {np.std(precisions):.4f}")

    # Average feature importances across runs
    importances_df = pd.concat(importances_list, axis=1).fillna(0.0)
    avg_s = importances_df.mean(axis=1)

    avg_importances = (
        avg_s.sort_values(ascending=False)
             .reset_index()
             .rename(columns={"index": "Feature", 0: "AvgImportance"})
    )

    print("\n=======================================")
    print("Average Feature Importance Across Runs")
    print("=======================================")
    print(avg_importances.head(20))

    return (
        confusions,
        scores,
        accuracies,
        recalls,
        precisions,
        y_tests,
        y_preds,
        avg_confusion,
        avg_importances
    )

In [ ]:
def optimize_random_forest_with_PCA_toggle_with_scaler_toggle_final(
    X, y, scaler_toggle=False, random_state=4, printing=False
):
    """
    Trains a RandomForestClassifier with n_estimators fixed to 500.
    Optionally applies StandardScaler (generally unnecessary for RF, but kept as a toggle).
    Returns the fitted pipeline, test split, and feature importances.
    """
    # Ensure y is 1D
    if hasattr(y, "to_numpy"):   # pandas Series/DataFrame
        y = y.to_numpy()
    y = np.asarray(y).ravel()    # (n_samples,)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    # Build pipeline (no PCA, no search)
    steps = []
    if scaler_toggle:
        steps.append(("scaler", StandardScaler()))
    steps.append(("rf", RandomForestClassifier(
        n_estimators=200,          # 150–300 is usually the sweet spot for speed/stability
        max_depth=18,              # cap depth to avoid huge trees 
        min_samples_leaf=2,        # reduces noisy splits, speeds up
        min_samples_split=8,       # fewer split evaluations, speeds up
        max_features="sqrt",       # critical for many features; reduces work per split
        bootstrap=True,
        oob_score=True,            # quick sanity metric without CV
        n_jobs=6,
        random_state=42
    )))
    model = Pipeline(steps)

    # Fit
    model.fit(X_train, y_train)

    # Predict / evaluate
    y_pred = model.predict(X_test)

    if printing:
        print("Model: RandomForestClassifier(n_estimators=500)")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

    # Feature importances
    rf_model = model.named_steps["rf"]
    feature_importances = pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    if printing:
        print("\nTop 10 Most Important Features:")
        print(feature_importances.head(10))

    return model, X_test, y_test, feature_importances

In [ ]:
sampling_data_100 = pd.read_csv('../100_combined_samples.csv')
sampling_data_100 = sampling_data_100.loc[:, (sampling_data_100 != 0).any(axis=0)]

In [ ]:
sampling_data_100["Unnamed: 0"] = sampling_data_100["Unnamed: 0"].str.extract(r"(FAC\d+)")
sampling_data_100

In [ ]:
sampling_data_100 = sampling_data_100.rename(
    columns={"Unnamed: 0": "FAC_ID"}
)
mouse_names = sampling_data_100["FAC_ID"]


In [ ]:
mouse_names = mouse_names.to_frame(name="FAC_ID")

In [ ]:
sampling_data_100 = sampling_data_100.drop(columns=["FAC_ID"])

In [ ]:
optimize_random_forest_with_PCA_toggle_with_scaler_toggle_final(X = sampling_data_100, y = mouse_names, printing=True)

In [ ]:
# Convert the DataFrame to a 1D Series before passing to the function
mouse_names_series = mouse_names.iloc[:, 0]  # Extract the first (and only) column as a Series
del mouse_names

In [ ]:
# Convert mouse labels to a 1D Series for classification
mouse_names_series = mouse_names.iloc[:, 0]

# Run mouse identity prediction
results_individual_mice = repeat_rf_optimization_with_scaler_toggle_final(
    X=sampling_data_100,
    y=mouse_names_series,
    n_runs=100,
    random_state=1337,
    scaler_toggle=False
)

In [ ]:
(confusions_individuals, scores_individuals, accuracies_individuals, recalls_individuals, precisions_individuals,
 y_tests_individuals, y_preds_individuals,
 avg_confusion_individuals, avg_importances_individuals) = results_individual_mice

# save per-run confusion matrices
np.save("../final_result/individual_prediction/confusions_individuals_final.npy", np.array(confusions_individuals))

# Instead of np.save, stack them into a readable CSV that keeps indices.
print("Stacking predictions for safe storage...")
all_preds_rows = []
for run_id, (yt, yp) in enumerate(zip(y_tests_individuals, y_preds_individuals), start=1):
    # Create a mini-dataframe for this run; yt.index preserves the Sample ID
    df_run = pd.DataFrame({
        "y_true": yt.values,
        "y_pred": yp.values if hasattr(yp, "values") else yp,
        "run": run_id
    }, index=yt.index) # <--- This safeguards Sample/Mouse IDs
    all_preds_rows.append(df_run)

# Concatenate and save
all_predictions_df = pd.concat(all_preds_rows)
all_predictions_df.to_csv("../final_result/individual_prediction/all_predictions_individuals_detailed_final.csv", index=True)

# save average confusion matrix
pd.DataFrame(avg_confusion_individuals).to_csv("../final_result/individual_prediction/avg_confusion_individuals_final.csv", index=False)

# save average feature importances (if PCA off)
if avg_importances_individuals is not None:
    avg_importances_individuals.to_csv("../final_result/individual_prediction/avg_importances_individuals_final.csv", index=False)

# Hard individuals

In [ ]:
import seaborn as sns

## Diet

In [ ]:
pred_long_diet = pred_long.copy()

In [ ]:
pred_long_diet.head()

In [ ]:
def generate_hard_sample_report(diet_predictions, threshold=0.35):
    """
    Analyzes prediction stability to find consistently misclassified MICE.
    
    Args:
        diet_predictions: DataFrame with columns ['y_true', 'y_pred', 'run', 'mouse_id']
        threshold: Error rate cutoff (0.35 means wrong >35% of the time)
    """
    df = diet_predictions.copy()
    
    # Flag errors
    df['is_error'] = df['y_true'] != df['y_pred']
    
    # Aggregate by 'mouse_id'
    # We calculate the error rate across ALL runs and ALL samples for that mouse
    mouse_stats = df.groupby('mouse_id').agg(
        Error_Rate=('is_error', 'mean'),
        True_Class=('y_true', 'first'), # Assumes a mouse doesn't change class!
        Most_Common_Pred=('y_pred', lambda x: x.mode()[0] if not x.mode().empty else None),
        Total_Observations=('run', 'count')
    )
    
    # Filter for "Hard" mice
    hard_mice = mouse_stats[mouse_stats['Error_Rate'] > threshold].sort_values(
        by='Error_Rate', ascending=False
    )
    
    return hard_mice

# Usage 
hard_mice_report = generate_hard_sample_report(pred_long_diet, threshold=0.5)
print(hard_mice_report.head(10))
hard_mice_report.to_csv("../final_result/diet_pred_flux/hard_mice_report.csv")

In [ ]:
def analyze_hard_individuals(individual_preds, threshold=0.10):
    """
    Analyzes which Individuals are consistently misidentified.
    
    Args:
        individual_preds: DataFrame with columns Error_Rate, True_Class, Most_Common_Pred, Total_Observations
        threshold: Error rate cutoff (e.g., 0.10 means wrong >10% of the time)
    """
    df = individual_preds[individual_preds['Error_Rate'] > threshold].sort_values(by='Error_Rate', ascending=False).reset_index(drop=True)

    # --- PLOTTING ---
    if not df.empty:
        plt.figure(figsize=(12, 6))
        
        # Create Bar Plot of Error Rates
        sns.barplot(
            data=df,
            x='mouse_id',
            y='Error_Rate',
            color='salmon',
            edgecolor='black'
        )
        
        plt.ylabel("Error Rate (0.0 - 1.0)", fontsize=22)
        plt.xlabel("Individual ID", fontsize=22)
        plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold})')
        plt.xticks(rotation=45)
        #plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Good news! No individuals had an error rate above {threshold}.")


# Set a lower threshold (e.g., 0.05) because individual ID is usually very accurate
hard_mice_report = analyze_hard_individuals(hard_mice_report, threshold=0.5)

## Individuals

In [ ]:
merged_preds = pd.read_csv("../final_result/individual_prediction/all_predictions_individuals_detailed_final.csv", index_col=0)

In [ ]:
merged_preds

In [ ]:
# Extract the columns directly from the merged DataFrame
# These contain the data for ALL runs stacked together
y_true = merged_preds['y_true'].to_numpy()
y_pred = merged_preds['y_pred'].to_numpy()

sample_ids = merged_preds.index.to_numpy()

print(f"Extracted {len(y_true)} samples.")
print(f"Example y_true: {y_true[:5]}")
print(f"Example y_pred: {y_pred[:5]}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_hard_individuals(individual_preds, threshold=0.10):
    """
    Analyzes which Individuals are consistently misidentified.
    
    Args:
        individual_preds: DataFrame with columns ['y_true', 'y_pred', 'run']
                          (Here, 'y_true' is the Actual Individual ID)
        threshold: Error rate cutoff (e.g., 0.10 means wrong >10% of the time)
    """
    df = individual_preds.copy()
    
    # 1. Flag Errors: Did we predict the wrong Individual?
    df['is_error'] = df['y_true'] != df['y_pred']
    
    # 2. Aggregate by Individual (y_true)
    # We want to know: For mouse FAC001, how often did we fail to recognize it?
    indiv_stats = df.groupby('y_true').agg(
        Error_Rate=('is_error', 'mean'),
        Most_Common_Confusion=('y_pred', lambda x: x[x != x.name].mode()[0] if not x[x != x.name].empty else "None"),
        Total_Observations=('run', 'count')
    ).reset_index()
    
    # Rename column for clarity
    indiv_stats.rename(columns={'y_true': 'Individual_ID'}, inplace=True)
    
    # 3. Filter for "Hard" Individuals (Error Rate > Threshold)
    hard_individuals = indiv_stats[indiv_stats['Error_Rate'] > threshold].sort_values(
        by='Error_Rate', ascending=False
    )
    
    # --- PLOTTING ---
    if not hard_individuals.empty:
        plt.figure(figsize=(12, 6))
        
        # Create Bar Plot of Error Rates
        sns.barplot(
            data=hard_individuals,
            x='Individual_ID',
            y='Error_Rate',
            color='salmon',
            edgecolor='black'
        )
        
        plt.title(f"Hardest-to-Identify Individuals (Error Rate > {threshold})", fontsize=15)
        plt.ylabel("Error Rate (0.0 - 1.0)", fontsize=12)
        plt.xlabel("Individual ID", fontsize=12)
        plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold})')
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Good news! No individuals had an error rate above {threshold}.")
        
    return hard_individuals

# --- USAGE ---
# Assuming 'pred_long_individuals' is your dataframe from the individual prediction task
# columns: ['y_true', 'y_pred', 'run'] where y_true is the Mouse ID (e.g., 'FAC001')

# Set a lower threshold (e.g., 0.05) because individual ID is usually very accurate
hard_indiv_report = analyze_hard_individuals(merged_preds, threshold=0.05)

print("Top 10 Hardest Individuals:")
print(hard_indiv_report.head(10))

# Save to CSV
hard_indiv_report.to_csv("../final_result/hard_individuals_report.csv", index=False)

In [ ]:
def analyze_hard_individuals_detailed(individual_preds, threshold=0.10):
    """
    Identifies individuals with high error rates and analyzes what they're confused with.
    
    Parameters:
    -----------
    individual_preds : DataFrame
        Predictions with columns ['y_true', 'y_pred', 'run']
    threshold : float
        Error rate threshold to flag as "hard" individual
    
    Returns:
    --------
    DataFrame with error stats and confusion details for hard individuals
    """
    df = individual_preds.copy()
    
    # Calculate error rate per individual (using all data)
    df['is_error'] = df['y_true'] != df['y_pred']
    
    stats = df.groupby('y_true').agg(
        Error_Rate=('is_error', 'mean'),
        Total_Observations=('run', 'count')
    ).reset_index()
    stats.rename(columns={'y_true': 'Individual_ID'}, inplace=True)
    
    # Analyze confusions: for each individual, what are they most confused with?
    # Focus only on rows where prediction was wrong
    errors_df = df[df['is_error'] == True]
    
    def get_confusion_summary(sub_df):
        """For a group of errors, return the most common confusion and top 3 details."""
        if sub_df.empty:
            return pd.Series([None, None], index=['Most_Common_Confusion', 'Confusion_Details'])
        
        # Count which individuals are being predicted (all wrong predictions)
        counts = sub_df['y_pred'].value_counts()
        
        # Most common wrong prediction
        top_1 = counts.index[0]
        
        # Format top 3: "FAC005 (45), FAC009 (12)"
        summary_str = ", ".join([f"{idx} ({val})" for idx, val in counts.head(3).items()])
        
        return pd.Series([top_1, summary_str], index=['Most_Common_Confusion', 'Confusion_Details'])

    # Apply confusion analysis to each individual's errors
    confusion_stats = errors_df.groupby('y_true').apply(get_confusion_summary).reset_index()
    confusion_stats.rename(columns={'y_true': 'Individual_ID'}, inplace=True)
    
    # Merge error rates and confusion details (left join: keep all individuals)
    full_report = pd.merge(stats, confusion_stats, on='Individual_ID', how='left')
    
    # Fill NaNs for individuals with no errors
    full_report['Most_Common_Confusion'].fillna("None", inplace=True)
    full_report['Confusion_Details'].fillna("None", inplace=True)
    
    # Filter for high-error individuals and sort by error rate (descending)
    hard_individuals = full_report[full_report['Error_Rate'] > threshold].sort_values(
        by='Error_Rate', ascending=False
    )
    
    # Plot results
    if not hard_individuals.empty:
        plt.figure(figsize=(12, 6))
        
        # Bar chart of error rates
        ax = sns.barplot(
            data=hard_individuals,
            x='Individual_ID',
            y='Error_Rate',
            color='salmon',
            edgecolor='black'
        )
        
        # Annotate bars with the most common wrong prediction
        for p, label in zip(ax.patches, hard_individuals['Most_Common_Confusion']):
            ax.annotate(f"Confused w/\n{label}", 
                        (p.get_x() + p.get_width() / 2., p.get_height()), 
                        ha='center', va='bottom', fontsize=6, color='black', xytext=(0, 5), 
                        textcoords='offset points')

        plt.title(f"Hardest Individuals (Error Rate > {threshold})\nAnnotations show most common wrong prediction", fontsize=14)
        plt.ylabel("Error Rate", fontsize=12)
        plt.xlabel("Individual ID", fontsize=12)
        plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold ({threshold})')
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Good news! No individuals had an error rate above {threshold}.")
        
    return hard_individuals


# Usage example
hard_indiv_report = analyze_hard_individuals_detailed(merged_preds, threshold=0.05)

print("Detailed Confusion Report:")
print(hard_indiv_report[['Individual_ID', 'Error_Rate', 'Confusion_Details']].head(10))
hard_indiv_report.to_csv("../final_result/hard_indiv_for_figure.csv", index=False)

## If hard individuals share a diet/sex/age/strain

### diet

In [ ]:
diet_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Diet')

In [ ]:
diet_data

In [ ]:
merged_preds

In [ ]:
# Create lookup: class ID → diet
diet_lookup = diet_data.drop_duplicates("FAC_ID").set_index("FAC_ID")["Diet"]

# Map class IDs to diet
df = merged_preds.copy()
df["diet_true"] = df["y_true"].map(diet_lookup)
df["diet_pred"] = df["y_pred"].map(diet_lookup)

# Check for unmapped diet values
missing_true = df["diet_true"].isna().mean()
missing_pred = df["diet_pred"].isna().mean()
print("Missing diet (true):", missing_true)
print("Missing diet (pred):", missing_pred)

# Keep only rows with valid diet mappings
df = df.dropna(subset=["diet_true", "diet_pred"])


In [ ]:
# Flag misclassified samples
df["is_error"] = df["y_true"] != df["y_pred"]

# Analyze errors: did model predict same diet or different diet?
err = df[df["is_error"]].copy()
err["same_diet"] = err["diet_true"] == err["diet_pred"]

# Overall: what fraction of errors are within same diet?
within_diet_error_rate = err["same_diet"].mean()
n_errors = len(err)

print("Total errors:", n_errors)
print("Within-diet fraction among errors:", within_diet_error_rate)


In [ ]:
# Errors per run: within-diet fraction and count
per_run = err.groupby("run")["same_diet"].agg(["mean","count"]).rename(
    columns={"mean":"within_diet_frac","count":"n_errors"}
).reset_index()

print(per_run.sort_values("within_diet_frac", ascending=False).to_string(index=False))

In [ ]:
# Errors per true diet: within-diet fraction and count
by_true_diet = err.groupby("diet_true")["same_diet"].agg(["mean","count"]).rename(
    columns={"mean":"within_diet_frac","count":"n_errors"}
)
print(by_true_diet)


In [ ]:
# Baseline: what's the probability of same-diet errors if random?
# If errors were random, P(same diet) = sum(p_i^2) where p_i = fraction of samples in diet i
diet_freq = df["diet_true"].value_counts(normalize=True)
p_same_baseline = (diet_freq**2).sum()

print("Diet frequencies:\n", diet_freq)
print("Baseline expected same-diet probability:", p_same_baseline)
print("Observed within-diet fraction among errors:", within_diet_error_rate)

# If observed > baseline: model confuses similar diets more often
# If observed ≈ baseline: errors are random
# If observed < baseline: model tends to misclassify across diets


### sex

In [ ]:
sex_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Sex')

In [ ]:
# Create lookup: class ID → sex
sex_lookup = (
    sex_data.drop_duplicates("FAC_ID")
    .set_index("FAC_ID")["Sex"]
)

# Map class IDs to sex
df = merged_preds.copy()
df["sex_true"] = df["y_true"].map(sex_lookup)
df["sex_pred"] = df["y_pred"].map(sex_lookup)

# Check for unmapped sex values
missing_true = df["sex_true"].isna().mean()
missing_pred = df["sex_pred"].isna().mean()
print("Missing sex (true):", missing_true)
print("Missing sex (pred):", missing_pred)

# Keep only rows with valid sex mappings
df = df.dropna(subset=["sex_true", "sex_pred"])
df["is_error"] = df["y_true"] != df["y_pred"]

# Analyze errors: did model predict same sex or different sex?
err = df[df["is_error"]].copy()
err["same_sex"] = err["sex_true"] == err["sex_pred"]

# Overall: what fraction of errors are within same sex?
within_sex_error_rate = err["same_sex"].mean()
n_errors = len(err)

print("Total errors:", n_errors)
print("Within-sex fraction among errors:", within_sex_error_rate)

# Errors per run: within-sex fraction and count
per_run = err.groupby("run")["same_sex"].agg(["mean", "count"]).rename(
    columns={"mean": "within_sex_frac", "count": "n_errors"}
).reset_index()

print(per_run.sort_values("within_sex_frac", ascending=False).to_string(index=False))

# Errors per true sex: within-sex fraction and count
by_true_sex = err.groupby("sex_true")["same_sex"].agg(["mean", "count"]).rename(
    columns={"mean": "within_sex_frac", "count": "n_errors"}
)
print(by_true_sex)

# Baseline: what's the probability of same-sex errors if random?
# If errors were random, P(same sex) = sum(p_i^2) where p_i = fraction of samples in sex category i
sex_freq = df["sex_true"].value_counts(normalize=True)
p_same_baseline = (sex_freq**2).sum()

print("Sex frequencies:\n", sex_freq)
print("Baseline expected same-sex probability:", p_same_baseline)
print("Observed within-sex fraction among errors:", within_sex_error_rate)

# If observed > baseline: model confuses sexes more often (biased misclassification)
# If observed ≈ baseline: errors are random with respect to sex
# If observed < baseline: model tends to misclassify across sexes

In [ ]:
sex_freq = df["sex_true"].value_counts(normalize=True)
p_same_sex_baseline = (sex_freq ** 2).sum()

print("Sex frequencies:\n", sex_freq)
print("Baseline expected same-sex probability:", p_same_sex_baseline)
print("Observed within-sex fraction among errors:", within_sex_error_rate)


### age

In [ ]:
age_data = pd.read_csv("../age_info.csv", index_col=0)

In [ ]:
age_data

In [ ]:
# Create lookup: class ID → age group
age_lookup = (
    age_data.drop_duplicates("FAC_ID")
    .set_index("FAC_ID")["Age_Group"]
)

# Map class IDs to age groups
df = merged_preds.copy()
df["age_true"] = df["y_true"].map(age_lookup)
df["age_pred"] = df["y_pred"].map(age_lookup)

# Check for unmapped age groups
missing_true = df["age_true"].isna().mean()
missing_pred = df["age_pred"].isna().mean()
print("Missing age group (true):", missing_true)
print("Missing age group (pred):", missing_pred)

# Keep only rows with valid age group mappings
df = df.dropna(subset=["age_true", "age_pred"])
df["is_error"] = df["y_true"] != df["y_pred"]

# Analyze errors: did model predict same age group or different age group?
err = df[df["is_error"]].copy()
err["same_age_group"] = err["age_true"] == err["age_pred"]

# Overall: what fraction of errors are within same age group?
within_age_error_rate = err["same_age_group"].mean()
n_errors = len(err)

print("Total errors:", n_errors)
print("Within-age-group fraction among errors:", within_age_error_rate)

# Errors per run: within-age-group fraction and count
per_run = err.groupby("run")["same_age_group"].agg(["mean", "count"]).rename(
    columns={"mean": "within_age_group_frac", "count": "n_errors"}
).reset_index()

print(per_run.sort_values("within_age_group_frac", ascending=False).to_string(index=False))

# Errors per true age group: within-age-group fraction and count
by_true_age = err.groupby("age_true")["same_age_group"].agg(["mean", "count"]).rename(
    columns={"mean": "within_age_group_frac", "count": "n_errors"}
)
print(by_true_age)

# Baseline: what's the probability of same-age-group errors if random?
# If errors were random, P(same age group) = sum(p_i^2) where p_i = fraction of samples in age group i
age_freq = df["age_true"].value_counts(normalize=True)
p_same_baseline = (age_freq**2).sum()

print("Age-group frequencies:\n", age_freq)
print("Baseline expected same-age-group probability:", p_same_baseline)
print("Observed within-age-group fraction among errors:", within_age_error_rate)

# If observed > baseline: model confuses similar age groups more often
# If observed ≈ baseline: errors are random
# If observed < baseline: model tends to misclassify across age groups


### genotype

In [ ]:
genotype_data = get_condition("../aData_S1_AllOmicsandPhenotypeData.csv", row_name='Strain')

In [ ]:
genotype_data

In [ ]:
# Create lookup: class ID → strain name
strain_lookup = (
    genotype_data.drop_duplicates("FAC_ID")
    .set_index("FAC_ID")["Strain"]
)

# Map class IDs to strain names
df = merged_preds.copy()
df["strain_true"] = df["y_true"].map(strain_lookup)
df["strain_pred"] = df["y_pred"].map(strain_lookup)

# Check for unmapped strains
missing_true = df["strain_true"].isna().mean()
missing_pred = df["strain_pred"].isna().mean()
print("Missing strain (true):", missing_true)
print("Missing strain (pred):", missing_pred)

# Keep only rows with valid strain mappings
df = df.dropna(subset=["strain_true", "strain_pred"])
df["is_error"] = df["y_true"] != df["y_pred"]

# Analyze errors: did model predict same strain or different strain?
err = df[df["is_error"]].copy()
err["same_strain"] = err["strain_true"] == err["strain_pred"]

# Overall: what fraction of errors are within same strain?
within_strain_error_rate = err["same_strain"].mean()
n_errors = len(err)

print("Total errors:", n_errors)
print("Within-strain fraction among errors:", within_strain_error_rate)

# Errors per run: within-strain fraction and count
per_run = err.groupby("run")["same_strain"].agg(["mean", "count"]).rename(
    columns={"mean": "within_strain_frac", "count": "n_errors"}
).reset_index()

print(per_run.sort_values("within_strain_frac", ascending=False).to_string(index=False))

# Errors per true strain: within-strain fraction and count
by_true_strain = err.groupby("strain_true")["same_strain"].agg(["mean", "count"]).rename(
    columns={"mean": "within_strain_frac", "count": "n_errors"}
)
print(by_true_strain)

# Baseline: what's the probability of same-strain errors if random?
# If errors were random, P(same strain) = sum(p_i^2) where p_i = fraction of samples in strain i
strain_freq = df["strain_true"].value_counts(normalize=True)
p_same_baseline = (strain_freq**2).sum()

print("Strain frequencies:\n", strain_freq)
print("Baseline expected same-strain probability:", p_same_baseline)
print("Observed within-strain fraction among errors:", within_strain_error_rate)

# If observed > baseline: model confuses similar strains more often
# If observed ≈ baseline: errors are random
# If observed < baseline: model tends to misclassify across strains

# Visualizing the result

In [ ]:
def plot_confusion_matrix_with_variability(conf_matrices, class_names=None):
    """
    Plot the average confusion matrix across runs and show variability
    in each cell as mean ± standard deviation.

    Inputs:
        conf_matrices : list or array-like
            Collection of confusion matrices with the same shape, typically
            one confusion matrix per run.

        class_names : list, optional
            Names of the classes to use as x- and y-axis tick labels.
            If None, default labels are used.

    Returns:
        None
            Displays a heatmap of the mean confusion matrix with each cell
            annotated as mean ± standard deviation.
    """

    # Compute the mean and standard deviation across all confusion matrices
    mean_cm = np.mean(conf_matrices, axis=0)
    std_cm = np.std(conf_matrices, axis=0)

    # Build annotation labels in the format "mean ± std" for each cell
    labels = []
    rows, cols = mean_cm.shape
    for i in range(rows):
        row_labels = []
        for j in range(cols):
            text = f"{mean_cm[i, j]:.1f}\n±{std_cm[i, j]:.1f}"
            row_labels.append(text)
        labels.append(row_labels)
    labels = np.array(labels)

    # Plot the average confusion matrix as a heatmap
    plt.figure(figsize=(8, 6))

    ax = sns.heatmap(
        mean_cm,
        annot=labels,
        fmt='',
        cmap='Blues',
        xticklabels=class_names if class_names is not None else 'auto',
        yticklabels=class_names if class_names is not None else 'auto',
        linewidths=1,
        linecolor='black',
        cbar_kws={'label': 'Average Count'},
        annot_kws={"fontsize": 16}
    )

    # Label the axes
    ax.set_xlabel('Predicted Label', fontsize=22)
    ax.set_ylabel('True Label', fontsize=22)

    # Adjust tick label size
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)

    # Format the colorbar
    cbar = ax.collections[0].colorbar
    cbar.ax.set_ylabel('Average Count', fontsize=18)
    cbar.ax.tick_params(labelsize=16)

    plt.tight_layout()
    plt.show()


## diet

In [ ]:
confusion_matrix_diet = np.load("../final_result/diet_pred_flux/confusions_diet.npy") # read confusion matrices per run

In [ ]:
plot_confusion_matrix_with_variability(confusion_matrix_diet, class_names=["CD", "HF"]) # make the confusion matrix

In [ ]:
metrics_per_run_mouse_level = pd.read_csv("../final_result/metrics_per_run_mouse_level.csv") # read the metrics per mouse
confusion_matrix_diet_mouse_level = np.load("../final_result/confusions_diet_mouse_level.npy") #  read confusion matrixes per mouse

In [ ]:
plot_confusion_matrix_with_variability(confusion_matrix_diet_mouse_level, class_names=["CD", "HF"]) # plot the matrix

## individuals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LogNorm
from matplotlib.patches import Patch

In [ ]:
confusions_individuals = pd.read_csv("../final_result/individual_prediction/avg_confusion_individuals_final.csv").values # read confusion matrix

In [ ]:
def plot_cm_correct_black_incorrect_red(
    cm,
    title="Confusion Matrix",
    tick_step=25,
    mask_zeros=True,
    log_scale=False,
    figsize=(16, 14),
    show_diagonal_ref=False,
):
    """
    Visualize a confusion matrix with correct predictions in black 
    and incorrect predictions in red.
    
    Parameters:
    -----------
    cm : array-like
        The confusion matrix (rows=predictions, cols=true labels)
    title : str
        Title for the plot
    tick_step : int
        Spacing between tick labels (0 to hide all ticks)
    mask_zeros : bool
        If True, don't color cells with zero values
    log_scale : bool
        If True, use logarithmic scale for color intensity
    figsize : tuple
        Figure size (width, height)
    show_diagonal_ref : bool
        If True, draw a line along the diagonal
    """
    
    # Convert input to numpy array and ensure float type for calculations
    cm = np.asarray(cm, dtype=float)
    rows, cols = cm.shape
    d = min(rows, cols)  # Size of diagonal (for square or rectangular matrices)

    # ========== SEPARATE CORRECT AND INCORRECT PREDICTIONS ==========
    # Create a zeros matrix to hold only diagonal values (correct predictions)
    correct = np.zeros_like(cm)
    correct[np.arange(d), np.arange(d)] = cm[np.arange(d), np.arange(d)]

    # Create a matrix with only off-diagonal values (incorrect predictions)
    incorrect = cm.copy()
    incorrect[np.arange(d), np.arange(d)] = 0.0  # Remove diagonal

    # ========== APPLY MASKING (OPTIONAL) ==========
    # Masked arrays allow us to treat zeros specially (don't color them)
    if mask_zeros:
        correct_m = np.ma.masked_where(correct == 0, correct)
        incorrect_m = np.ma.masked_where(incorrect == 0, incorrect)
    else:
        # If not masking, create regular masked arrays with no mask applied
        correct_m = np.ma.array(correct, mask=False)
        incorrect_m = np.ma.array(incorrect, mask=False)

    # ========== CREATE NORMALIZATION FUNCTION ==========
    def make_norm(masked_arr):
        """
        Create a normalization object that maps values to [0, 1] for coloring.
        Handles both linear and logarithmic scales.
        """
        # Extract non-masked values
        vals = masked_arr.compressed()
        
        # For log scale, remove zeros and negatives (undefined in log space)
        if log_scale:
            vals = vals[vals > 0]
        
        # If no values to normalize, return None
        if vals.size == 0:
            return None
        
        # Create appropriate normalizer
        if log_scale:
            # LogNorm: values are scaled logarithmically
            return LogNorm(vmin=vals.min(), vmax=vals.max())
        else:
            # Linear Norm: 0 → white, max value → full intensity
            return Normalize(vmin=0, vmax=vals.max() if vals.max() > 0 else 1)

    # Build normalizers for correct and incorrect predictions
    norm_c = make_norm(correct_m)
    norm_i = make_norm(incorrect_m)

    # ========== COMPUTE COLOR INTENSITIES ==========
    # Create matrices to store normalized intensity values (0 to 1)
    intensity_c = np.zeros_like(cm)
    intensity_i = np.zeros_like(cm)

    # Apply normalization to map values to [0, 1] range
    if norm_c is not None:
        intensity_c = norm_c(correct_m).filled(0.0)  # Fill masked values with 0
    if norm_i is not None:
        intensity_i = norm_i(incorrect_m).filled(0.0)

    # ========== BUILD RGB COLOR ARRAY ==========
    # Start with white background (all channels = 1.0)
    rgb = np.ones((rows, cols, 3), dtype=float)

    # Correct predictions (black): subtract equally from R, G, B channels
    # High intensity_c → darker black; zero intensity_c → white
    rgb[..., 0] -= intensity_c  # Red channel
    rgb[..., 1] -= intensity_c  # Green channel
    rgb[..., 2] -= intensity_c  # Blue channel

    # Incorrect predictions (red): subtract from G and B only
    # High intensity_i → brighter red; zero intensity_i → white
    rgb[..., 1] -= intensity_i  # Green channel
    rgb[..., 2] -= intensity_i  # Blue channel

    # Clip to valid range [0.0, 1.0] to prevent invalid color values
    rgb = np.clip(rgb, 0.0, 1.0)

    # ========== CREATE AND CONFIGURE PLOT ==========
    fig, ax = plt.subplots(figsize=figsize)
    
    # Display the RGB array as an image
    # 'nearest' interpolation: no smoothing between pixels
    ax.imshow(rgb, interpolation="nearest", aspect="auto")

    # Force the matrix visualization to be square (1:1 aspect ratio)
    ax.set_box_aspect(1)

    # ========== CONFIGURE TICK MARKS ==========
    if tick_step and tick_step > 0:
        # Show tick marks at regular intervals
        ax.set_xticks(np.arange(0, cols, tick_step))
        ax.set_yticks(np.arange(0, rows, tick_step))
    else:
        # Hide all tick marks
        ax.set_xticks([])
        ax.set_yticks([])

    # ========== ADD LABELS AND TITLE ==========
    ax.set_xlabel("True label", fontsize=18)
    ax.set_ylabel("Predicted label", fontsize=18)

    if title:
        ax.set_title(title, fontsize=18)

    # ========== OPTIONAL: DIAGONAL REFERENCE LINE ==========
    if show_diagonal_ref:
        # Draw a line along the main diagonal (correct predictions)
        m = min(rows, cols)
        ax.plot(np.arange(m), np.arange(m), linewidth=1)

    # ========== ADD LEGEND ==========
    legend_handles = [
        Patch(facecolor="black", edgecolor="black", label="Correct (diagonal)"),
        Patch(facecolor="red", edgecolor="red", label="Incorrect (off-diagonal)")
    ]

    ax.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),  # Position to the right of plot
        frameon=True,
        fontsize=14
    )

    # ========== ADJUST LAYOUT ==========
    # Leave room on the right (82%) for the legend
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()

In [ ]:
# Run the plot
plot_cm_correct_black_incorrect_red(
    confusions_individuals,
    tick_step=25,
    mask_zeros=True,
    log_scale=False,
    title=False
)